In [1]:
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
import numpy as np
import os
import shutil
import zipfile
from sklearn.metrics import precision_score, recall_score, f1_score

In [2]:
from google.colab import drive
drive.mount('/content/drive_new')

Drive already mounted at /content/drive_new; to attempt to forcibly remount, call drive.mount("/content/drive_new", force_remount=True).


In [3]:
ZIP_FILES_FOLDER = "/content/drive_new/MyDrive/Hackathon/Samples/"
EXTRACTED_FOLDER = "/content/drive_new/MyDrive/Hackathon/content/extracted_images"       

os.makedirs(EXTRACTED_FOLDER, exist_ok=True)

In [4]:
for zip_filename in os.listdir(ZIP_FILES_FOLDER):
    if zip_filename.endswith('.zip'):
        zip_path = os.path.join(ZIP_FILES_FOLDER, zip_filename)
        extract_subfolder = os.path.join(EXTRACTED_FOLDER, os.path.splitext(zip_filename)[0])
        os.makedirs(extract_subfolder, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_subfolder)
        print(f"Extracted {zip_filename} to {extract_subfolder}")

Extracted Timber_h1.zip to /content/drive_new/MyDrive/Hackathon/content/extracted_images/Timber_h1
Extracted RCC_OS_H4.zip to /content/drive_new/MyDrive/Hackathon/content/extracted_images/RCC_OS_H4
Extracted RCC_OS_H3.zip to /content/drive_new/MyDrive/Hackathon/content/extracted_images/RCC_OS_H3
Extracted RCC_OS_H2.zip to /content/drive_new/MyDrive/Hackathon/content/extracted_images/RCC_OS_H2
Extracted RCC_OS_H1.zip to /content/drive_new/MyDrive/Hackathon/content/extracted_images/RCC_OS_H1
Extracted RCC_H6.zip to /content/drive_new/MyDrive/Hackathon/content/extracted_images/RCC_H6
Extracted RCC_H5.zip to /content/drive_new/MyDrive/Hackathon/content/extracted_images/RCC_H5
Extracted RCC_H4.zip to /content/drive_new/MyDrive/Hackathon/content/extracted_images/RCC_H4
Extracted RCC_H3.zip to /content/drive_new/MyDrive/Hackathon/content/extracted_images/RCC_H3
Extracted RCC_H2 gable roof.zip to /content/drive_new/MyDrive/Hackathon/content/extracted_images/RCC_H2 gable roof
Extracted RCC_H2 f

In [4]:
model = ResNet50(weights='imagenet', include_top=False, pooling='avg')

SIMILARITY_THRESHOLD = 0.85

In [ ]:
for root, dirs, files in os.walk(EXTRACTED_FOLDER):
    
    if "duplicates" in root:
        continue
    print(f"\nProcessing folder: {root}")
    duplicates_folder = os.path.join(root, "duplicates")
    os.makedirs(duplicates_folder, exist_ok=True)

    features_list = []
    filenames_list = []

    for filename in files:
        file_path = os.path.join(root, filename)
        try:
            img_raw = tf.io.read_file(file_path)
            img = tf.image.decode_image(img_raw, channels=3)
            img = tf.image.resize(img, (224, 224))
            img = preprocess_input(img)
            img = tf.expand_dims(img, axis=0)

            features = model(img)
            features = tf.linalg.l2_normalize(features, axis=1).numpy().flatten()
        except Exception as e:
            print(f"Error processing {filename} in {root}: {e}")
            continue

        duplicate_found = False
        for stored_feat, stored_filename in zip(features_list, filenames_list):
            similarity = np.dot(features, stored_feat)

            print(f"Comparing {filename} with {stored_filename}: Similarity = {similarity:.3f}")

            if similarity > SIMILARITY_THRESHOLD:
                duplicate_found = True
                print(f"🚨 Duplicate Found: {filename} is similar to {stored_filename} (Similarity: {similarity:.3f})")
                destination_path = os.path.join(root, "duplicates", filename)
                shutil.move(file_path, destination_path)
                print(f"Moved {filename} -> {destination_path}")
                break

        if not duplicate_found:
            features_list.append(features)
            filenames_list.append(filename)

    print(f"Deduplication completed for folder: {root}")



Streaming output truncated to the last 5000 lines.
Comparing 32.24242902_76.33216229_5255__9358-2.jpg with 32.23267923_76.32424065_4561__7561-1.jpg: Similarity = 0.549
Comparing 32.24242902_76.33216229_5255__9358-2.jpg with 32.23272487_76.32520117_5607__7575-1.jpg: Similarity = 0.563
Comparing 32.24242902_76.33216229_5255__9358-2.jpg with 32.23274851_76.32410191_1503__7587-2.jpg: Similarity = 0.587
Comparing 32.24242902_76.33216229_5255__9358-2.jpg with 32.23276275_76.32586723_1702__7593-1.jpg: Similarity = 0.609
Comparing 32.24242902_76.33216229_5255__9358-2.jpg with 32.23280549_77.19648973_910__4643-1.jpg: Similarity = 0.684
Comparing 32.24242902_76.33216229_5255__9358-2.jpg with 32.23283971_76.32591856_5404__7611-2.jpg: Similarity = 0.521
Comparing 32.24242902_76.33216229_5255__9358-2.jpg with 32.23283971_76.32591856_5404__7611-4.jpg: Similarity = 0.699
Comparing 32.24242902_76.33216229_5255__9358-2.jpg with 32.23290726_76.32600098_5407__7625-3.jpg: Similarity = 0.632
Comparing 32.2